In [58]:
import numpy as np
import pandas as pd
import sys
sys.path.append('/Users/moilanen/sbx/paragon/lib')
sys.path.append('/Users/moilanen/sbx/paragon/data')
sys.path.append('/Users/moilanen/sbx/paragon/data/provider')
sys.path.append('.')

import math
from datetime import datetime, timedelta, date

from support_resistance import sr

# stocks = ['QQQ', 'AMZN']
# stocks = ['IWF', 'MTUM', 'USMV', 'VYM', 'IWD', 'IJR']
# stocks = ['TSLA']
stocks = [
        'AAL',
        #'AAPL',
        'ABBV',
        'ADBE',
        'AMD',
        'AMZN',
        'AVGO',
        'BA',
        'BABA',
        'BAC',
        'BYND',
        'CAT',
        'CGC',
        'CLF',
        'COST',
        'CRM',
        'DBX',
        'DIS',
        'DKNG',
        'DOCU',
        'EDIT',
        'ETSY',
        'F',
        'FB',
        'FXI',
        'GOOGL',
        'HD',
        'HON',
        'IQ',
        'INTC',
        'IWM',
        'JD',
        'LULU',
        'LLY',
        'LYFT',
        'MSFT',
        'MU',
        'NFLX',
        'NIO',
        'NKE',
        'NKLA',
        'NOW',
        'NVDA',
        'ORCL',
        'PENN',
        'PINS',
        'PFE',
        'PM',
        'PTON',
        'PYPL',
        'QCOM',
        'QQQ',
        'ROKU',
        'SBUX',
        'SHOP',
        'SLB',
        'SNAP',
        'SPY',
        'SPCE',
        'SQ',
        'TDOC',
        'TLRY',
        # 'TSLA',
        'TTD',
        'TTWO',
        'TWLO',
        'TWTR',
        'UA',
        'UBER',
        'WMT',
        'WYNN',
        'XLF',
        'ZM',
        'ZNGA'
                  ]

# stocks = ['AAL', 'ADBE', 'AMD', 'BABA', 'C', 'AMZN', 'GOOGL', 'MSFT', 'SPCE', 'FB', 'ARKK', 'SLV', 'ZM','TWTR']


num_days = 500

start_date = date.today() - timedelta(days = math.ceil((num_days/(5/7))))


In [59]:
from polygon import RESTClient
from dateutil import tz
def aggregates_to_df(resp):
    epoch = datetime(1601, 1, 1)
    df = pd.DataFrame(resp.results)
#     tz_ = tz.gettz('America/New_York')
    tz_ = tz.gettz('UTC')

    df.set_index(df['t'].apply(lambda x: datetime.fromtimestamp(x / 1000, tz=tz_)), inplace=True)
    df.rename(columns={"v": "volume", "o": "open", "c": "close", "h": "high", "l": "low"}, inplace=True)
    if 't' in df:
        df.drop(['t'], axis=1, inplace=True)
    if 'n' in df:
        df.drop(['n'], axis=1, inplace=True)

    del df.index.name

    return df

def history(symbol, start_date, timespan = 'day', multiplier = 1):
    date_format = "%Y-%m-%d"
    
    end_date = date.today() + timedelta(days = 7) # Polygon is exclusive on dates and must extend for a week
    client = RESTClient('WeYC_RTF_nZg3S_UKGI68lvNZ__8l6YM09p_Rg')

    
    try:
        resp = client.stocks_equities_aggregates(symbol, multiplier, timespan, start_date.strftime(date_format), end_date.strftime(date_format))
    except Exception as e:
        print("[history_aggregate()]: Polygon connection error: %s" % (str(e)))

    if resp.status != 'OK':
        raise ValueError("Polygon history_aggregate response not OK: %s" % (resp['status']))
        
    if resp.results == None or len(resp.results) == 0:
        raise ValueError("Polygon history_aggregate results are empty")
        
    df = aggregates_to_df(resp)
    
    return df

from datetime import datetime
from dateutil import tz
import pandas_market_calendars as mcal
import time
class Polygon():
    date_format = "%Y-%m-%d"
    approximate_holes_active = False
    def __init__(self):
        self.mcal = mcal.get_calendar('NYSE')
        self.client = RESTClient('WeYC_RTF_nZg3S_UKGI68lvNZ__8l6YM09p_Rg')
    
    def history_aggregate(self, symbol, bar_count, frequency, start = False, end = False, after_hours = False):
        max_timespan = 5000
        converts = {
            '1m': {
                'timespan': 'minute',
                'multiplier': 1,
                'per_day': 390.0,
                'max_days': math.floor(max_timespan / 390.0),
                'granularity_minutes': 1
            },
            '5m': {
                'timespan': 'minute',
                'multiplier': 5,
                'per_day': 390 / 5.0,
                'max_days': math.floor(max_timespan / (390.0 / 5.0)),
                'granularity_minutes': 5
            },
            '20m': {
                'timespan': 'minute',
                'multiplier': 20,
                'per_day': 390 / 20.0,
                'max_days': math.floor(max_timespan / (390.0)),
                'granularity_minutes': 20
            },
            '30m': {
                'timespan': 'minute',
                'multiplier': 30,
                'per_day': 390 / 30.0,
                'max_days': math.floor(max_timespan / (390.0)),
                'granularity_minutes': 30
            },
            '1h': {
                'timespan': 'hour',
                'multiplier': 1,
                'per_day': 6.5,
                'max_days': math.floor(max_timespan / 24),
                'granularity_minutes': 60
            },
            '1d': {
                'timespan': 'day',
                'multiplier': 1,
                'per_day': 1.0,
                'max_days': max_timespan,
                'granularity_minutes': 24 * 60
            },
            '1w': {
                'timespan': 'week',
                'multiplier': 1,
                'per_day': 1 / 5.0,
                'max_days': math.floor(max_timespan * 7),
                'granularity_minutes': 24 * 60 * 7
            },
        }

        convert = converts[frequency]
        multiplier = convert['multiplier']
        timespan = convert['timespan']
        max_days = convert['max_days']
        granularity_minutes = convert['granularity_minutes']

        incomplete = False

        if end is False:
            end = self.get_datetime()

        if start is False:
            # See how many days we span
            days = (bar_count / convert['per_day']) + 1.0 # Make sure we go over
            days = days * 1.15

            start = self.mcal.get_days_ago(end, int(days))

        wanted_times_all = self.wanted_times(start, end, granularity_minutes, timespan)
        wanted_times_last = wanted_times_all[-1]

        # Polygon is exclusive dates
        end = end + timedelta(days=1)
        need_data = True
        start_ = datetime(start.year, start.month, start.day, tzinfo=self.timezone())
        df_out = False
        attempts = 0

        # Pdb().set_trace()

        while need_data:
            if attempts == 0:
                end_ = min(end, (start_ + timedelta(days=max_days)))
            try:
#                 self.debug("[%s]: Calling start: %s end: %s Attempt: %d" % (symbol, str(start_), str(end_), attempts))
                resp = self.client.stocks_equities_aggregates(symbol, multiplier, timespan, start_.strftime(self.date_format), end_.strftime(self.date_format))
            except Exception as e:
                print("[history_aggregate()]: Polygon connection error: %s" % (str(e)))
                time.sleep(3)
                continue

            if resp.status != 'OK':
                raise ValueError("Polygon history_aggregate response not OK: %s" % (resp['status']))

            if resp.results == None or len(resp.results) == 0:
                #print("Empty Results: Start: %s End: %s" % (start_, end_))
                start_ = end_

                # This only happens if there is no data
                if start_ >= self.get_datetime():
                    # print("[%s]: No Data at all" % (symbol))
                    return df_out

                continue

            df = self.aggregates_to_df(resp)

            df_orig = df
            # print(str(df))
            if after_hours is False:
                # Pdb().set_trace()
                df = self.after_hours_remove(df, timespan)

            if df_out is False:
                df_out = df
            else:
                df_out = pd.concat([df_out, df], sort=True)
            df_out = df_out.loc[~df_out.index.duplicated(keep='first')]
            df_out = df_out.sort_index()

            wanted_times = self.wanted_times(start_, end_, granularity_minutes, timespan)

            missing = set(wanted_times).difference(df.index.to_list())
            if len(missing) > 0 and attempts < 1:
                first = min(missing)
                if start_ == first.to_pydatetime():

                    print("[%s] Missing[%d]: %s :: %s - %s :: Attempt: %d" % (symbol, len(missing), str(first), str(start_), str(end_), attempts))

                    # Try breaking the cache of polygon to get a different answer
                    #end_ = min(end, (start_ + timedelta(days=random.randint(1, 3))))
                    # start_ = first.to_pydatetime() - timedelta(days=random.randint(1, 3))
                    # print("[%s] Trying: %s - %s" % (symbol, str(start_), str(end_)))

                    attempts += 1
                    time.sleep(0.5 * attempts)
                else:
                    start_ = first.to_pydatetime()
                continue
            if len(missing) > 0 and attempts >= 1:
                incomplete = True
            if len(missing) > 0 and attempts >= 1 and self.approximate_holes_active is True:
                df_out = self.approximate_holes(df_out, missing, symbol)

            attempts = 0

            start_ = df_orig.iloc[-1].name.to_pydatetime()

            # print("%s :: %s" % (str(wanted_times_last.strftime(self.date_format)), str(start_.strftime(self.date_format))))
            # print("%s :: %s" % (str((end - timedelta(days=1)).strftime(self.date_format)), str(start_.strftime(self.date_format))))
            # if (end - timedelta(days=1)).strftime(self.date_format) == start_.strftime(self.date_format):
            if wanted_times_last.strftime(self.date_format) <= start_.strftime(self.date_format):
                need_data = False

        # Drop duplicates
        df_out = df_out.loc[~df_out.index.duplicated(keep='first')]

        missing_all = set(wanted_times_all).difference(df_out.index.to_list())
        if len(missing_all) > 0:
            # print("[%s] Missing Data: %s" % (symbol, str(len(missing_all))))
            pass

        if len(df_out) < bar_count:
            print("Polygon history_aggregate does not have enough data. Want: %d Have: %d DF: %s" % (bar_count, len(df_out), str(df_out)))
            # Pdb().set_trace()
            # raise ValueError("Polygon history_aggregate does not have enough data. Want: %d Have: %d DF: %s" % (bar_count, len(df), str(df)))

        return df_out.sort_index().iloc[-bar_count:,].dropna(how='all')


    def aggregates_to_df(self, resp):
        epoch = datetime(1601, 1, 1)
        df = pd.DataFrame(resp.results)
        tz = self.timezone()
        df.set_index(df['t'].apply(lambda x: datetime.fromtimestamp(x / 1000, tz=tz)), inplace=True)
        df.rename(columns={"v": "volume", "o": "open", "c": "close", "h": "high", "l": "low"}, inplace=True)
        if 't' in df:
            df.drop(['t'], axis=1, inplace=True)
        if 'n' in df:
            df.drop(['n'], axis=1, inplace=True)

        del df.index.name

        return df
    
    def timezone(self):
        return tz.gettz('America/New_York')
    def get_datetime(self):
        return datetime.now(self.timezone())
    
    def wanted_times(self, start, end, delta, timespan):
        mcal = self.mcal
        out = []

        if end > self.get_datetime():
            end = self.get_datetime()

        convert = {
            'minute': 'min',
            'hour': 'hour',
            'day': 'D',
            'week': 'W'
        }

        dti_timespan = convert[timespan]

        schedule = mcal.schedule(start, end)
        schedule['market_open'] = schedule['market_open'].dt.tz_convert(self.timezone())
        schedule['market_close'] = schedule['market_close'].dt.tz_convert(self.timezone())

        for index, row in schedule.iterrows():

            day_start = row['market_open']
            if timespan in ['day', 'week']:
                day_start = day_start.replace(hour = 0, minute = 0, second = 0, microsecond = 0)
                if timespan == 'day':
                    td = timedelta(days=delta)
                else:
                    td = timedelta(days=(delta*7))
            else:
                td = timedelta(minutes=delta)
                day_start = row['market_open']
                if timespan == 'hour':
                    day_start = day_start.replace(hour = 9, minute = 0, second = 0, microsecond = 0)
                else:
                    minute = (30 % delta) + 30
                    day_start = day_start.replace(hour = 9, minute = minute, second = 0, microsecond = 0)

            if row['market_close'] > self.get_datetime():
                day_end = self.get_datetime() - timedelta(minutes = 1)
            else:
                day_end = row['market_close']

            dts = [pd.Timestamp(dt) for dt in self.datetime_range(day_start, day_end, td)]

            out = out + dts

        return out
    
    
    def datetime_range(self, start, end, delta):
        current = start
        while current < end:
            yield current
            current += delta
            
    def after_hours_remove(self, df, timespan):
        if timespan in ['day', 'week']:
            return df
        tz = self.timezone()
        if timespan == 'hour':
            df = df.between_time('9:00', '16:00', include_end = False)
        else:
            df = df.between_time('9:30', '16:00', include_start = True, include_end = False)

        if len(df) == 0:
            return pd.DataFrame()

        start_date = df.iloc[0].name
        end_date = df.iloc[-1].name
        delta = end_date - start_date
        days_between = delta.days
        assert days_between >= 0, "Unexpected order in after_hours_remove: %d" % (days_between)

        sched = self.mcal.schedule(end_date, days_between)
        early_closes = self.mcal.early_closes(sched)

        for index, row in early_closes.iterrows():
            # Premarket drops
            to_drop = df[df.index.date == index.date()].between_time('00:00', row['market_open'].astimezone(tz).to_pydatetime().replace(tzinfo=tz).time(), include_end = False)
            if len(to_drop):
                df = df.drop(to_drop.index)

            # After Hours
            to_drop = df[df.index.date == index.date()].between_time(row['market_close'].astimezone(tz).to_pydatetime().replace(tzinfo=tz).time(), '23:59:59', include_start = False)
            if len(to_drop):
                df = df.drop(to_drop.index)

        return df



In [60]:
# Pull Data

ohlcvs = {}
ohlcvs_long = {}

polygon = Polygon()

if False:
    timeframe_short = '5m'
    periods_day_short = int(390/5)
else:
    timeframe_short = '30m'
    periods_day_short = int(390/30)

    
for stock in stocks:
    print("------ %s ------" % (stock))
    ohlcv = polygon.history_aggregate(stock, int(periods_day_short*num_days), timeframe_short, start_date, after_hours = True)
#     ohlcv = history(stock, start_date, timespan = "minute", multiplier = 30)
    ohlcvs[stock] = ohlcv

    ohlcv = ohlcv = polygon.history_aggregate(stock, num_days, '1d', start_date, after_hours = True)
    ohlcvs_long[stock] = ohlcv

#     print(str(ohlcvs_long))
# df = pd.DataFrame(ohlcvs)
# print(str(ohlcvs))
# sr(ohlcv, 14)

------ AAL ------
Polygon history_aggregate does not have enough data. Want: 500 Have: 483 DF:                                 volume       vw   open  close     high    low
2018-12-31 00:00:00-05:00    5334664.0  32.1086  32.09  32.11  32.6200  31.68
2019-01-02 00:00:00-05:00    5229460.0  32.2300  31.46  32.48  32.6500  31.05
2019-01-03 00:00:00-05:00   16821991.0  29.8033  31.69  30.06  31.8535  28.81
2019-01-04 00:00:00-05:00    9369633.0  31.6202  30.44  32.04  32.0900  30.40
2019-01-07 00:00:00-05:00    8010692.0  32.8022  31.99  32.95  33.4804  31.24
...                                ...      ...    ...    ...      ...    ...
2020-11-20 00:00:00-05:00   60605655.0  12.6148  12.80  12.53  12.9100  12.46
2020-11-23 00:00:00-05:00  100763995.0  13.1691  12.75  13.56  13.5800  12.69
2020-11-24 00:00:00-05:00  156500970.0  14.6463  14.32  14.82  14.9700  14.08
2020-11-25 00:00:00-05:00  101052472.0  14.7959  14.77  14.94  15.0800  14.37
2020-11-27 00:00:00-05:00   66066370.0  15.2458

Polygon history_aggregate does not have enough data. Want: 500 Have: 483 DF:                                volume        vw    open   close      high  \
2018-12-31 00:00:00-05:00   3557891.0  321.3380  320.50  322.50  323.6400   
2019-01-02 00:00:00-05:00   3292308.0  320.5961  316.19  323.81  323.9500   
2019-01-03 00:00:00-05:00   5705817.0  312.8373  319.49  310.90  319.7400   
2019-01-04 00:00:00-05:00   4448790.0  324.3099  316.69  327.08  328.4400   
2019-01-07 00:00:00-05:00   4030385.0  327.6445  330.52  328.11  330.6900   
...                               ...       ...     ...     ...       ...   
2020-11-20 00:00:00-05:00  36406623.0  202.0374  204.59  199.62  206.5800   
2020-11-23 00:00:00-05:00  28519342.0  208.8964  203.24  211.53  214.2099   
2020-11-24 00:00:00-05:00  31503618.0  219.4634  219.43  218.49  222.9500   
2020-11-25 00:00:00-05:00  19308462.0  217.2727  217.71  217.61  221.0000   
2020-11-27 00:00:00-05:00  10135301.0  217.2428  218.25  216.50  219.9300   

Polygon history_aggregate does not have enough data. Want: 500 Have: 483 DF:                                volume       vw   open  close    high    low
2018-12-31 00:00:00-05:00   7989812.0   7.6395   7.92   7.69   7.960   7.46
2019-01-02 00:00:00-05:00   6366759.0   7.7761   7.49   7.86   7.960   7.44
2019-01-03 00:00:00-05:00   7097961.0   7.6793   7.76   7.63   7.865   7.49
2019-01-04 00:00:00-05:00  12418185.0   8.3657   7.83   8.47   8.640   7.80
2019-01-07 00:00:00-05:00  12220096.0   8.5727   8.55   8.73   8.820   8.17
...                               ...      ...    ...    ...     ...    ...
2020-11-20 00:00:00-05:00  29812942.0   9.2961   9.16   9.23   9.500   9.15
2020-11-23 00:00:00-05:00  17726813.0   9.7590   9.40   9.93   9.960   9.37
2020-11-24 00:00:00-05:00  23038152.0  10.7368  10.18  10.85  11.100  10.12
2020-11-25 00:00:00-05:00  13218194.0  10.9950  10.80  11.18  11.280  10.52
2020-11-27 00:00:00-05:00   7309883.0  11.3729  11.41  11.29  11.598  11.25

[483 rows 

Polygon history_aggregate does not have enough data. Want: 500 Have: 152 DF:                            close   high      low     open      volume       vw
2020-04-24 00:00:00-04:00  19.35  20.75  17.6000  20.4937  11475270.0  19.3634
2020-04-27 00:00:00-04:00  19.52  22.50  19.5000  22.3600   9594187.0  20.7023
2020-04-28 00:00:00-04:00  18.24  19.90  18.0000  19.8600   5725702.0  18.6771
2020-04-29 00:00:00-04:00  19.40  19.80  18.3500  18.3500   3649405.0  19.2982
2020-04-30 00:00:00-04:00  19.46  19.85  18.8800  19.5000   3974807.0  19.4332
...                          ...    ...      ...      ...         ...      ...
2020-11-20 00:00:00-05:00  48.23  50.05  47.3900  49.7900  25839706.0  48.5064
2020-11-23 00:00:00-05:00  48.03  48.88  46.9000  48.5000  18142528.0  47.8784
2020-11-24 00:00:00-05:00  47.89  49.04  46.6100  48.3464  17707567.0  48.1003
2020-11-25 00:00:00-05:00  50.23  51.08  47.7700  47.7900  23223062.0  50.0335
2020-11-27 00:00:00-05:00  52.75  52.98  50.7069  51.8

Polygon history_aggregate does not have enough data. Want: 500 Have: 483 DF:                               volume         vw     open    close       high  \
2018-12-31 00:00:00-05:00  1655504.0  1043.4439  1057.83  1044.96  1062.9900   
2019-01-02 00:00:00-05:00  1593395.0  1046.5385  1027.20  1054.68  1060.7900   
2019-01-03 00:00:00-05:00  2097957.0  1033.4499  1050.67  1025.47  1066.2600   
2019-01-04 00:00:00-05:00  2301428.0  1068.5482  1042.56  1078.07  1080.0000   
2019-01-07 00:00:00-05:00  2372667.0  1074.0396  1080.97  1075.92  1082.7000   
...                              ...        ...      ...      ...        ...   
2020-11-20 00:00:00-05:00  1407579.0  1748.6432  1762.00  1736.38  1768.3625   
2020-11-23 00:00:00-05:00  1127618.0  1725.8775  1740.22  1727.56  1745.9900   
2020-11-24 00:00:00-05:00  1362116.0  1750.7255  1727.50  1763.90  1766.4750   
2020-11-25 00:00:00-05:00   979957.0  1763.0085  1767.81  1764.13  1770.3800   
2020-11-27 00:00:00-05:00   739507.0  1783.

...

Polygon history_aggregate does not have enough data. Want: 500 Have: 483 DF:                                volume       vw   open  close    high    low
2018-12-31 00:00:00-05:00  14932851.0  45.1756  45.19  45.15  45.500  44.84
2019-01-02 00:00:00-05:00  14321228.0  45.0603  44.48  45.22  45.340  44.45
2019-01-03 00:00:00-05:00  19869968.0  44.9924  44.75  44.78  45.500  44.41
2019-01-04 00:00:00-05:00  20984989.0  46.6000  45.37  46.71  46.950  45.25
2019-01-07 00:00:00-05:00  17971163.0  47.5628  46.93  47.45  48.105  46.47
...                               ...      ...    ...    ...     ...    ...
2020-11-20 00:00:00-05:00  16935846.0  56.0106  56.36  55.70  56.440  55.68
2020-11-23 00:00:00-05:00   8365770.0  56.0330  55.85  56.08  56.570  55.77
2020-11-24 00:00:00-05:00  10565666.0  57.3856  56.60  57.57  57.900  56.36
2020-11-25 00:00:00-05:00   6949546.0  57.4244  57.72  57.41  57.760  57.26
2020-11-27 00:00:00-05:00   5763208.0  57.6712  57.40  57.76  57.870  57.15

[483 rows 

Polygon history_aggregate does not have enough data. Want: 500 Have: 483 DF:                                volume        vw    open   close    high  \
2018-12-31 00:00:00-05:00   8877184.0   56.7935   57.12   56.91   57.41   
2019-01-02 00:00:00-05:00   9896584.0   57.3934   56.20   57.40   58.00   
2019-01-03 00:00:00-05:00  14422231.0   55.9536   55.95   55.70   56.84   
2019-01-04 00:00:00-05:00  14177270.0   56.3537   56.50   56.60   56.83   
2019-01-07 00:00:00-05:00  12351967.0   56.4899   56.39   56.44   57.16   
...                               ...       ...     ...     ...     ...   
2020-11-20 00:00:00-05:00   5625399.0  147.5257  148.28  146.03  149.10   
2020-11-23 00:00:00-05:00  11654921.0  143.5732  142.35  143.82  147.00   
2020-11-24 00:00:00-05:00  10807321.0  143.8417  143.39  145.93  146.33   
2020-11-25 00:00:00-05:00   6398910.0  144.9521  145.95  144.08  147.49   
2020-11-27 00:00:00-05:00   3878048.0  144.8910  145.49  143.83  146.43   

                      

[history_aggregate()]: Polygon connection error: HTTPSConnectionPool(host='api.polygon.io', port=443): Read timed out. (read timeout=None)
Polygon history_aggregate does not have enough data. Want: 500 Have: 483 DF:                                volume       vw   open  close   high      low
2018-12-31 00:00:00-05:00  15325338.0   5.5333   5.76   5.51   5.86   5.4200
2019-01-02 00:00:00-05:00  15253505.0   5.7272   5.38   5.79   5.87   5.3500
2019-01-03 00:00:00-05:00  16429337.0   5.7277   5.67   5.68   5.87   5.6300
2019-01-04 00:00:00-05:00  18200582.0   5.8468   5.67   5.95   5.99   5.6300
2019-01-07 00:00:00-05:00  14729901.0   6.1199   6.00   6.21   6.21   5.9200
...                               ...      ...    ...    ...    ...      ...
2020-11-20 00:00:00-05:00  57469475.0  43.5499  42.62  44.29  44.32  42.2200
2020-11-23 00:00:00-05:00  37597190.0  46.0909  47.10  45.26  47.27  45.1100
2020-11-24 00:00:00-05:00  24642579.0  44.3707  45.60  44.29  45.69  43.3618
2020-11-25 00:

Polygon history_aggregate does not have enough data. Want: 500 Have: 483 DF:                               volume        vw     open   close      high  \
2018-12-31 00:00:00-05:00  2169912.0  103.0285  105.060  102.94  105.7500   
2019-01-02 00:00:00-05:00  1947440.0  102.8154  100.240  104.01  104.5200   
2019-01-03 00:00:00-05:00  2301077.0  100.3926  102.400   99.15  103.0000   
2019-01-04 00:00:00-05:00  3610836.0  101.6942  101.130  101.70  103.7600   
2019-01-07 00:00:00-05:00  2501858.0  103.0137  101.500  104.41  104.9400   
...                              ...       ...      ...     ...       ...   
2020-11-20 00:00:00-05:00  1475092.0  170.5484  165.990  170.05  172.5300   
2020-11-23 00:00:00-05:00  1138935.0  169.7960  170.050  171.19  171.5900   
2020-11-24 00:00:00-05:00  1222728.0  169.6808  171.100  169.83  171.2656   
2020-11-25 00:00:00-05:00   889221.0  173.1488  170.550  173.98  174.4350   
2020-11-27 00:00:00-05:00   615069.0  176.1795  175.865  176.41  177.9400   

Polygon history_aggregate does not have enough data. Want: 500 Have: 483 DF:                                volume       vw   open  close    high      low
2018-12-31 00:00:00-05:00  56675821.0  23.6986  23.72  23.82  23.920  23.5200
2019-01-02 00:00:00-05:00  63044521.0  23.8928  23.47  24.02  24.070  23.4000
2019-01-03 00:00:00-05:00  65894230.0  23.6386  23.84  23.48  23.900  23.4200
2019-01-04 00:00:00-05:00  64744808.0  24.1205  23.88  24.26  24.310  23.8200
2019-01-07 00:00:00-05:00  48235552.0  24.3289  24.21  24.29  24.510  24.0600
...                               ...      ...    ...    ...     ...      ...
2020-11-20 00:00:00-05:00  38748247.0  27.2124  27.28  27.18  27.385  27.1218
2020-11-23 00:00:00-05:00  52326032.0  27.6220  27.53  27.69  27.770  27.3900
2020-11-24 00:00:00-05:00  60759518.0  28.4764  28.17  28.66  28.690  28.0950
2020-11-25 00:00:00-05:00  45282932.0  28.4636  28.50  28.56  28.580  28.2200
2020-11-27 00:00:00-05:00  27125478.0  28.4607  28.56  28.47  28.

In [61]:
ohlcvs['AAL']

,close,high,low,open,volume,vw
2020-02-04 07:00:00-05:00,27.540,27.5400,27.16,27.1600,6955.0,27.29137
2020-02-04 07:30:00-05:00,27.540,27.5500,27.45,27.5000,22010.0,27.52862
2020-02-04 08:00:00-05:00,27.600,27.6193,27.25,27.5002,12837.0,27.53718
2020-02-04 08:30:00-05:00,27.650,27.8400,27.44,27.4400,27283.0,27.66517
2020-02-04 09:00:00-05:00,27.640,27.7700,27.60,27.6200,7176.0,27.65085
...,...,...,...,...,...,...
2020-11-27 12:00:00-05:00,15.165,15.2650,15.15,15.2600,3666302.0,15.19280
2020-11-27 12:30:00-05:00,14.970,15.1700,14.95,15.1650,9364369.0,15.02622
2020-11-27 13:00:00-05:00,14.980,14.9800,14.98,14.9800,285084.0,14.98000
2020-11-27 16:00:00-05:00,15.010,15.0100,14.97,15.0000,54656.0,14.99528


# Backtest

In [65]:
import talib

def quantity_get(account_size, leverage_max, position_size, want, have, price, position_open = True):
    max_size = account_size * leverage_max
    

def backtest(stock, ohlcv, ohlcv_long, stop_loss_atrp = 1.5, price_target_multiple = 3, quantity_multiple = 1.5, stop_loss_room = 0.0):
    verbose = False
    
    profit_based_sl = True
    trail_stop_enabled = False
    start_size = 100000
    account_size = start_size
    position_size = 0.01
#     leverage_max = 1.20
    
    sma_small = int(9*(periods_day_short*1.5))
    sma_big = int(21*(periods_day_short*1.5))
    timeperiod = 14
    holdings = 0
    
    total_quantity = 0
    
#     print("Quantity: %d" % (quantity))
    
    total_open = 0.0
    total_close = 0.0
    
    pnl = 0.0
    pnl_pct = 0.0
    
#     import pdb; pdb.set_trace()

    ohlcv_long['atr'] = talib.ATR(ohlcv_long['high'], ohlcv_long['low'], ohlcv_long['close'], timeperiod = timeperiod)
    ohlcv_long['atr_std'] = ohlcv_long['atr'].rolling(timeperiod).std()
    idx = ohlcv_long.index.searchsorted(ohlcv.index, side='left') - 1
#     idx = idx[idx > 0]
    ohlcv['atr'] = pd.Series(ohlcv_long['atr'][idx].to_list(), index=ohlcv.index)
    ohlcv['atr_std'] = pd.Series(ohlcv_long['atr_std'][idx].to_list(), index=ohlcv.index)
    ohlcv['sma_small'] = ohlcv['close'].rolling(sma_small).mean()
    ohlcv['sma_big'] = ohlcv['close'].rolling(sma_big).mean()

    ohlcv['atrp'] = ohlcv['atr_std'] / ohlcv['close']
    ohlcv['adx'] = talib.ADX(ohlcv['high'], ohlcv['low'], ohlcv['close'])
    ohlcv = ohlcv.dropna(axis=0)
    
#     print(str(ohlcv))

    opened = False
    for index, bar in ohlcv.iloc[1:].iterrows():
        index_number = ohlcv.index.get_loc(index)
#         print(str(ohlcv.loc[index]))
#         print(str(index_number))
        big_prior = ohlcv['sma_big'].iloc[index_number - 1]
        small_prior = ohlcv['sma_small'].iloc[index_number - 1]
        big_current = ohlcv['sma_big'].iloc[index_number]
        small_current = ohlcv['sma_small'].iloc[index_number]
        
        price = bar['close']
        
#         print("%0.2f :: %0.2f" % (big_current, small_current))

        
        # Open
        if opened is False and big_prior >= small_prior and big_current < small_current:
#             import pdb; pdb.set_trace()
            quantity = ((account_size * position_size) / ohlcv.iloc[0]['open'])
            opened = price
            holdings += quantity * opened
            total_quantity += quantity
            
            stop_loss_distance = (bar['atrp'] * stop_loss_atrp)
            price_target_distance = stop_loss_distance * price_target_multiple

            stop_loss = opened * (1.0 - stop_loss_distance)
            price_target = opened * (1.0 + price_target_distance)
            if verbose:
                print("%s [%s] Opened: %0.2f PT: %0.2f SL: %0.2f  ATRP: %0.2f%%" % (index, stock, opened, price_target, stop_loss, bar['atrp']*100.0))
            continue

        if opened is not False:
#             import pdb; pdb.set_trace()
            profit = (total_quantity * price) - holdings
            profit_distance = profit * stop_loss_room
            
            
            # Close Stop Loss
#             import pdb; pdb.set_trace()
            sma_low = ohlcv['low'].iloc[index_number - 5: index_number].mean()
            if bar['low'] <= stop_loss and bar['high'] >= price_target:
                continue
                print("%s Warning: Have a bar that spans both PT: %0.2f and SL %0.2f" % (index, price_target, stop_loss))
                print((str(bar)))

#             if sma_low <= stop_loss:

            if bar['low'] <= stop_loss:
#                 if verbose:
#                     print("%s [%s] SL Hit: %0.2f <= %0.2f :: " % (index, stock, stop_loss, bar['low']))
#                 import pdb; pdb.set_trace()
                holdings_close = (stop_loss * total_quantity)

                total_open += holdings
                total_close += holdings_close

#                 if verbose:
#                     print("open: %0.2f close: %0.2f" % (total_open, total_close))
            
                pnl += (holdings_close - holdings)
                pnl_pct += (holdings_close - holdings) / holdings
            

                account_size += (holdings_close - holdings)
                if verbose:
                    print("%s [%s] PnL: %0.2f%%:: SL Hit: %0.2f <= %0.2f Holdings: %0.2f Avg Price: %0.2f Shares: %d" % (index, stock, ((account_size - start_size) / start_size) * 100.0, bar['low'], stop_loss, holdings, holdings / total_quantity, total_quantity))

                opened = False
                total_quantity = 0
                holdings = 0.0
                trail_stop_enabled = False

            elif bar['high'] >= price_target:
#                 import pdb; pdb.set_trace()
                if verbose:
                    print("PT Hit: high: %0.2f pt: %0.2f" % (bar['high'], price_target))
                trail_stop_enabled = True

                if True:
                    opened = max(price_target, bar['open'])
                    quantity_available = (account_size - holdings) / opened
                    quantity_want = quantity * quantity_multiple
                    
                    quantity = int(min(quantity_available, quantity_want))
                    
                    holdings += quantity * opened
                    total_quantity += quantity

                    if profit_based_sl:
                        pass
#                         stop_loss = opened - (profit_distance / total_quantity)
                    else:
                        stop_loss = opened * (1.0 - stop_loss_room)

                    price_target = opened * (1.0 + price_target_distance)
                    if price_target < opened:
                        import pdb; pdb.set_trace()
                else:
                    pnl += (bar['high'] * total_quantity) - holdings
                    pnl_pct += ((bar['high'] * total_quantity) - holdings) / holdings

                    opened = False
                    total_quantity = 0
                    holdings = 0.0
                    
                if verbose:
                    print("%s [%s] PT Hit: %0.2f >= %0.2f :: Avg Price: %0.2f Shares: %d" % (index, stock, bar['high'], opened, holdings / total_quantity, total_quantity))

            if trail_stop_enabled: # Trail Stop
                if True: # Profit based Stop Loss
#                     import pdb; pdb.set_trace()
                    stop_loss_candidate = bar['high'] - (profit_distance / total_quantity)
                else:
                    stop_loss_candidate = bar['high'] * (1.0 - (stop_loss_distance))

                if (stop_loss < stop_loss_candidate):
                    stop_loss_orig = stop_loss
                    stop_loss = stop_loss_candidate

                    if verbose:
                        print("%s SL Increase: %0.2f -> %0.2f" % (index, stop_loss_orig, stop_loss))
                    
                    continue



    if total_open > 0:
        pnl_pct = (total_close - total_open) / total_open
#         pnl_pct = (total_close - total_open) / start_size
        pnl_pct_account_size = (account_size - start_size) / start_size
#         print("[%s]\t\tFinal PnL: Account: %0.2f%% Trades: %0.2f%%" % (stock, pnl_pct_account_size * 100.0, pnl_pct*100.0))
#         return pnl_pct
#         return total_close - total_open
        return pnl_pct_account_size
    else:
        return 0.0
        


# Pyramid

In [63]:
def run(params):#, stop_loss_atrp, price_target_multiple, quantity_multiple):
#     import pdb; pdb.set_trace()
    stocks = params['stocks']
    ohlcvs = params['ohlcv']
    ohlcvs_long = params['ohlcv_long']
    stop_loss_atrp = params['stop_loss_atrp']
    price_target_multiple = params['price_target_multiple']
    quantity_multiple = params['quantity_multiple']
    stop_loss_room = params['stop_loss_room']
#     import pdb; pdb.set_trace()
#     print(str(params))
    pnls = []
    
    print("Stop Loss ATRP: %0.2f Price Target Multiple: %0.2f Quantity Multiple: %0.2f Stop Loss Room: %0.2f" % (stop_loss_atrp, price_target_multiple, quantity_multiple, stop_loss_room))
    

    
    res = {}
    for stock in stocks:
        ohlcv = ohlcvs[stock]
        ohlcv_long = ohlcvs_long[stock]
        pnl_pct = backtest(stock, ohlcv, ohlcv_long, stop_loss_atrp = stop_loss_atrp, price_target_multiple = price_target_multiple, quantity_multiple = quantity_multiple, stop_loss_room = stop_loss_room)
        pnls.append(pnl_pct)
        res[stock] = pnl_pct
    
    pnls = np.array(pnls)
    pnl_pct = np.mean(pnls)
    pnl_median = np.median(pnls)
    pnl_std = np.std(pnls)

    sharpe = pnl_pct / pnl_std
#     print(str(res))
    print("PnL Percent Final: %0.2f%% Median: %0.2f%% Std: %0.2f Sharpe: %0.2f" % (pnl_pct*100.0, pnl_median*100.0, pnl_std, sharpe))
    
    return pnl_pct, pnl_median, sharpe

In [66]:
import skopt

SPACE = [skopt.space.Real(1.0, 2.5, name='stop_loss_atrp', prior='uniform'),
         skopt.space.Real(0.5, 5.0, name='price_target_multiple', prior='uniform'),
         skopt.space.Real(0.01, 2.0, name='quantity_multiple', prior='uniform'),
         skopt.space.Real(0.01, 0.5, name='stop_loss_room', prior='uniform'),
         ]

N_ROWS=10000
STATIC_PARAMS = {'stocks': stocks, 'ohlcv': ohlcvs, 'ohlcv_long': ohlcvs_long}
HPO_PARAMS = {'n_calls':100,
              'n_random_starts':10,
              'base_estimator':'ET',
              'acq_func':'EI',
              'xi':0.02,
              'kappa':1.96,
              'n_points':10000,
             }

experiment_params = {**STATIC_PARAMS, 
                     **HPO_PARAMS,
                     'n_rows': N_ROWS
                    }

@skopt.utils.use_named_args(SPACE)
def objective(**params):
    all_params = {**params, **STATIC_PARAMS}
    pnl_mean, pnl_median, sharpe = run(all_params)    
    return -1.0 * pnl_mean


results = skopt.forest_minimize(objective, SPACE, **HPO_PARAMS)
best_auc = -1.0 * results.fun
best_params = results.x


    
# log metrics
print('Best Validation AUC: {}'.format(best_auc))
print('Best Params: {}'.format(best_params))

# 

Stop Loss ATRP: 2.26 Price Target Multiple: 4.03 Quantity Multiple: 0.63 Stop Loss Room: 0.35
PnL Percent Final: 0.03% Median: 0.01% Std: 0.00 Sharpe: 0.29
Stop Loss ATRP: 1.70 Price Target Multiple: 1.38 Quantity Multiple: 0.70 Stop Loss Room: 0.20
PnL Percent Final: 0.07% Median: 0.04% Std: 0.00 Sharpe: 0.62
Stop Loss ATRP: 2.50 Price Target Multiple: 3.85 Quantity Multiple: 0.28 Stop Loss Room: 0.05
PnL Percent Final: 0.03% Median: 0.01% Std: 0.00 Sharpe: 0.34
Stop Loss ATRP: 1.39 Price Target Multiple: 2.69 Quantity Multiple: 1.28 Stop Loss Room: 0.25
PnL Percent Final: 0.06% Median: 0.04% Std: 0.00 Sharpe: 0.62
Stop Loss ATRP: 1.45 Price Target Multiple: 4.80 Quantity Multiple: 0.02 Stop Loss Room: 0.29
PnL Percent Final: 0.02% Median: 0.01% Std: 0.00 Sharpe: 0.25
Stop Loss ATRP: 2.38 Price Target Multiple: 4.48 Quantity Multiple: 1.72 Stop Loss Room: 0.02
PnL Percent Final: 0.06% Median: 0.02% Std: 0.00 Sharpe: 0.44
Stop Loss ATRP: 1.59 Price Target Multiple: 2.09 Quantity Multip

PnL Percent Final: 0.07% Median: 0.02% Std: 0.00 Sharpe: 0.44
Stop Loss ATRP: 1.25 Price Target Multiple: 1.91 Quantity Multiple: 0.93 Stop Loss Room: 0.37
PnL Percent Final: 0.06% Median: 0.03% Std: 0.00 Sharpe: 0.62
Stop Loss ATRP: 1.39 Price Target Multiple: 0.91 Quantity Multiple: 1.57 Stop Loss Room: 0.31
PnL Percent Final: 0.21% Median: 0.08% Std: 0.00 Sharpe: 0.55
Stop Loss ATRP: 1.30 Price Target Multiple: 1.88 Quantity Multiple: 0.30 Stop Loss Room: 0.48
PnL Percent Final: 0.03% Median: 0.02% Std: 0.00 Sharpe: 0.57
Stop Loss ATRP: 1.79 Price Target Multiple: 4.76 Quantity Multiple: 0.89 Stop Loss Room: 0.43
PnL Percent Final: 0.03% Median: 0.02% Std: 0.00 Sharpe: 0.37
Stop Loss ATRP: 1.88 Price Target Multiple: 2.57 Quantity Multiple: 1.51 Stop Loss Room: 0.11
PnL Percent Final: 0.09% Median: 0.03% Std: 0.00 Sharpe: 0.50
Stop Loss ATRP: 2.40 Price Target Multiple: 4.35 Quantity Multiple: 0.28 Stop Loss Room: 0.18
PnL Percent Final: 0.03% Median: -0.00% Std: 0.00 Sharpe: 0.28
S

In [55]:
results
res_dict = {}
i = 0
while i < len(results.x_iters):
    res_dict[float(results.func_vals[i])] = results.x_iters[i]
    i += 1
# df_prev = df  
df = pd.DataFrame.from_dict(res_dict, orient='index')

df.sort_index(ascending=True).iloc[0:int(len(res)/10.0)].median()
print(str(df.sort_index(ascending=True).iloc[0:int(len(res)/10.0)]))
# res = sorted(res_dict.keys())
'''
Short 250 30 min
0    1.667956
1    4.147399
2    1.524218
3    0.115973

Full 250 30 min
0    2.023039
1    0.739520
2    1.940295
3    0.207080

                  0         1         2         3
-0.002275  2.332876  0.717291  1.989532  0.210264
-0.001810  1.509000  0.753260  1.968694  0.098929
-0.001785  2.120503  0.570346  1.673497  0.211263
-0.001635  1.925575  1.377690  1.988792  0.203896
-0.001564  2.308722  0.621069  1.761561  0.035840
-0.001564  2.326767  0.914634  1.963818  0.213231
-0.001457  2.282212  0.602886  1.592933  0.238353
-0.001443  1.180699  0.725780  1.494970  0.164517
-0.001438  1.231439  1.175122  1.962084  0.091527
-0.001424  1.715261  1.468148  1.918505  0.368514
'''
# keys = res[0:int(len(res)/5.0)]
# pd.Dataframe()

                  0         1         2         3
-0.002275  2.332876  0.717291  1.989532  0.210264
-0.001810  1.509000  0.753260  1.968694  0.098929
-0.001785  2.120503  0.570346  1.673497  0.211263
-0.001635  1.925575  1.377690  1.988792  0.203896
-0.001564  2.308722  0.621069  1.761561  0.035840
-0.001564  2.326767  0.914634  1.963818  0.213231
-0.001457  2.282212  0.602886  1.592933  0.238353
-0.001443  1.180699  0.725780  1.494970  0.164517
-0.001438  1.231439  1.175122  1.962084  0.091527
-0.001424  1.715261  1.468148  1.918505  0.368514


'\nShort 250 30 min\n0    1.667956\n1    4.147399\n2    1.524218\n3    0.115973\n\nFull 250 30 min\n0    2.023039\n1    0.739520\n2    1.940295\n3    0.207080\n'

In [16]:
params = {
    'stop_loss_atrp': 1.0,
    'price_target_multiple': 0.9,
    'quantity_multiple':2.0,
    'stop_loss_room': 0.4
    
}
STATIC_PARAMS = {'stocks': stocks, 'ohlcv': ohlcvs, 'ohlcv_long': ohlcvs_long}

all_params = {**params, **STATIC_PARAMS}

run(all_params)

Stop Loss ATRP: 1.00 Price Target Multiple: 0.90 Quantity Multiple: 2.00 Stop Loss Room: 0.40
PnL Percent Final: 0.00% Median: 0.00% Std: 0.00 Sharpe: 0.19


(2.45436878583236e-05, 4.042137478172662e-06, 0.19214478974095844)

Optimizing on % returns from start:
=========================================

with Trail Stop

PnL Percent Final: 32.52% Median: 29.22% Std: 0.30 Sharpe: 1.08
Best Validation AUC: 0.32517942512592074
Best Params: [1.8109885840654074, 4.134385801108951, 0.010006169345897365, 0.2937744300340398]

-----------------------------
w/o Std ATR

PnL Percent Final: 39.32% Median: 37.30% Std: 0.31 Sharpe: 1.25
Best Validation AUC: 0.39318588114697817
Best Params: [0.5286344469627664, 0.5187052045087182, 1.0608345265413026, 0.025425376122422468]

-----------------------------

Optimizing on median % returns from start:
==========================================================

with Trail Stop

PnL Percent Final: 32.73% Median: 29.65% Std: 0.30 Sharpe: 1.10

Best Validation AUC: 0.29648188841092865
Best Params: [1.5447711483005364, 4.573400715143107, 0.019796786939590093, 0.2893707310833107]


----------------------------

w/o Std ATR

PnL Percent Final: 38.% Median: 36.45% Std: 0.30 Sharpe: 1.27
Best Validation AUC: 0.3645056784932671
Best Params: [0.535138722087267, 0.518531100735852, 1.0500551372105824, 0.061291326683916646]


Optimizing on Sharpe:
=============================================================

with Trail Stop

PnL Percent Final: 10.07% Median: 11.90% Std: 0.08 Sharpe: 1.20
Best Validation AUC: 1.2049018566834817
Best Params: [2.2844321708372837, 4.817119381629968, 0.5685637521846277, 0.2533964708976639]

---------------------------------

w/o Trail Stop

---------------------------------


Optimizing on Total Returns:
==============================================================

with Trail Stop

PnL Percent Final: 198966.20% Median: 150998.85% Std: 511336.46 Sharpe: 0.39
Best Validation AUC: 198966.20458940996
Best Params: [2.2047403862139965, 3.3767040385266287, 1.796270054595376, 0.01411611184963516]


Optimizing on Median Total Returns:
==============================================================

with Trail Stop

PnL Percent Final: 87371.10% Median: 391161.49% Std: 207354.37 Sharpe: 0.42
Best Validation AUC: 3911.6148943241424
Best Params: [2.233759729607554, 2.3027564839785812, 1.9624906537885738, 0.4179681480257582]


==============================================




Universe:

Optimized on PnL %
No Trail Stop
ATR Timeperiod: 34

PnL Percent Final: 144.30% Std: 4.46 Sharpe: 0.32
Best Validation AUC: 1.5448901720780561
Best Params: [1.2522713428457701, 3.1971240071895597, 0.021764643984546667]

------------------------------
ATR Timeperiod: 14
PnL Percent Final: 148.66% Std: 4.53 Sharpe: 0.33
PnL Percent Final: 144.31% Std: 4.56 Sharpe: 0.32
PnL Percent Final: 26.85% Std: 0.35 Sharpe: 0.77
Best Validation AUC: 1.5139663572587045
Best Params: [1.2277032301835273, 1.8548135352318065, 0.012398609048573803]

--------------------------------
ATR Timeperiod: 14
Stop Loss: On
PnL Percent Final: 139.10% Std: 4.16 Sharpe: 0.33
Best Validation AUC: 1.3999460197378097
Best Params: [1.9467205252323778, 1.1978270609888177, 0.010490316384664312]

----------------------------------
Using median to optimize on

PnL Percent Final: 10.16% Median: 1.80% Std: 0.22 Sharpe: 0.45
Best Validation AUC: 0.029705617122962793
Best Params: [1.7809382039424666, 1.7333467819704238, 0.01798387029830515]

-----------------------------------
Median
Stop Loss raised - 20%

PnL Percent Final: 139.29% Median: 27.17% Std: 4.18 Sharpe: 0.33
Best Validation AUC: 0.2732928766506795
Best Params: [2.1859839639882153, 1.2148282576016207, 0.024914497287100545]

-------------------------------------
Median
Stop Loss raised - 20% & value is 5 SMA

PnL Percent Final: 28.63% Median: 26.08% Std: 0.34 Sharpe: 0.85
Best Validation AUC: 0.27036705677601447
Best Params: [1.9633119866593196, 0.5675435270199632, 0.026238205693981158]

---------------------------------------
Median
Stop loss is 5 SMA

PnL Percent Final: 29.48% Median: 26.25% Std: 0.34 Sharpe: 0.87
Best Validation AUC: 0.2716357015314919
Best Params: [1.9299624392054449, 0.5307301246992547, 0.012453130044499553]

----------------------------------------
Median
Stop Loss Raised - 30%

PnL Percent Final: 27.56% Median: 29.27% Std: 0.31 Sharpe: 0.88
Best Validation AUC: 0.29270543733379334
Best Params: [1.78287337591409, 2.381201410014458, 0.01183352950245321]

------------------------------------------
Median

PnL Percent Final: 24.71% Median: 25.23% Std: 0.30 Sharpe: 0.83
Best Validation AUC: 0.2522670422392403
Best Params: [2.3408935394442256, 1.972693067366336, 0.016783369481519775, 0.3457435246906491]

--------------------------
Give up percentage of profit only

PnL Percent Final: 22.16% Median: 22.99% Std: 0.18 Sharpe: 1.23
Best Validation AUC: 0.22989768849730147
Best Params: [2.353107930203058, 4.107175406905601, 0.12824169530923554, 0.03849591172279658]

----------------------------
Fix to use stop loss as exit point and not the low

PnL Percent Final: 26.30% Median: 21.72% Std: 0.20 Sharpe: 1.29
Best Validation AUC: 0.2575408226808793
Best Params: [2.3334009714883637, 4.159298952227813, 0.08449998151878182, 0.010961327743237386]

-----------------------------
Fix to use price target as open point

PnL Percent Final: 30.63% Median: 26.17% Std: 0.24 Sharpe: 1.28
Best Validation AUC: 0.26172042090680897
Best Params: [2.118553625967377, 4.475981339556505, 0.019963512833690904, 0.01560491032330384]

----------------------------------
Optimize on total made vs starting amount

PnL Percent Final: 4.68% Median: 0.92% Std: 14.53 Sharpe: 0.32
Best Validation AUC: 4.684595956502898
Best Params: [1.3213880833992722, 4.900181686516508, 1.981103301531843, 0.03892394323228026]
-----------------------------------
Optimize on median made vs starting amount

In [10]:
import pprint
pp = pprint.PrettyPrinter(indent=4)

portfolio_size = 3200000

for stock, weight in weights['weight'].sort_values(ascending=False).iteritems():
    amount = portfolio_size * weight
 
    print('{a}\t {b:5,.2f}%\t {c:10,.0f}'.format(a=stock, b=weight*100.0, c=amount))

GLD	 49.67%	  1,589,438
TLT	 35.62%	  1,139,694
QQQ	  5.72%	    182,984
ARKK	  5.38%	    172,209
USRT	  3.61%	    115,675
